# செலவு கோரிக்கை பகுப்பாய்வு

இந்த நோட்புக் பயணச் செலவுகளைக் கையாள உள்ளக ரசீதுப் படிமங்களைப் பயன்படுத்தி பிளகின்களைப் பயன்படுத்தும் முகவர்களை உருவாக்குவதையும், செலவுக் கோரிக்கை மின்னஞ்சலை உருவாக்குவதையும், மற்றும் செலவு தரவுகளை ஒரு பை வரைபடம் மூலம் காட்சிப்படுத்துவதையும் காட்டுகிறது. முகவர்கள் பணிபுரியும் சூழலின் அடிப்படையில் செயல்பாடுகளை தேர்ந்தெடுக்கின்றனர்.

படி:
1. OCR முகவர் உள்ளக ரசீதுப் படிமத்தை செயலாக்கி பயணச் செலவு தரவுகளை எடுக்கும்.
2. மின்னஞ்சல் முகவர் செலவு கோரிக்கை மின்னஞ்சலை உருவாக்கும்.

### பயணச் செலவு சூழலுக்கான எடுத்துக்காட்டு:
நீங்கள் மற்றொரு நகரத்திற்கு தொழிலாளர் கூட்டத்திற்கான பயணத்தில் இருப்பவர் என கற்பனை செய்யுங்கள். உங்கள் நிறுவனம் அனைத்து சாதாரண பயணச் செலவுகளையும் மீள்பணம் வழங்கும் கொள்கையை கொண்டுள்ளது. இதோ, சாத்தியமான பயணச் செலவுகளின் விவரங்கள்:
- போக்குவரத்து:
உங்கள் வீட்டிமாநகரத்திலிருந்து இலக்குமாநகரத்திற்கான சுற்றுலா விமான டிக்கெட்.
விசிற்றுக்கு வருவதற்கும் செல்லுவதற்கும் டேக்சி அல்லது ரைட்-ஹெய்லிங் சேவைகள்.
இலக்குமாநகரத்தில் உள்ளூர் போக்குவரத்து (பொதுப் போக்குவரத்து, வாடகை கார்கள், அல்லது டேக் சிகள் போன்றவை).

- வசதி:
மூன்று இரவுகளுக்கான நீண்ட நடுத்தர வணிக ஹோட்டல்ச் சேவை, கூட்டத் தளத்திற்கு அருகே.

- உணவுகள்:
தினசரி உணவு கூடு, காலை உணவு, மதிய உணவு மற்றும் இரவு உணவுக்கான செய்முறை, நிறுவனத்தின் நாள் பரிசீலனை கொள்கைக்கு ஏற்ப.

- வேறுபட்ட செலவுகள்:
விமான நிலையத்தில் பார்கிங் கட்டணங்கள்.
ஹோட்டலில் இணைய சேவை கட்டணங்கள்.
கை கொடுக்கல் அல்லது சிறிய சேவை கட்டணங்கள்.

- ஆவணப்படுத்தல்:
நீங்கள் அனைத்து ரசீதுகளையும் (விமானங்கள், டேக் சிகள், ஹோட்டல், உணவுகள், முதலியன) மற்றும் மீள்பணம் பெறுவதற்கான ஒரு முடிக்கப்பட்ட செலவு அறிக்கையையும் சமர்ப்பிக்கின்றீர்கள்.


## தேவையான நூலகங்களை இறக்குமதி செய்யவும்

உரையாடல் புத்தகத்திற்கான தேவையான நூலகங்கள் மற்றும் தொகுதிகளை இறக்குமதி செய்யவும்.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## செலவுத் மாதிரிகளை வரையறுக்கவும்

 தனிப்பட்ட செலவுகளுக்கு Pydantic மாதிரியை உருவாக்கவும் மற்றும் பயனர் கேள்வியை கட்டமைக்கப்பட்ட செலவு தரவுகளாக மாற்ற ExpenseFormatter வகுப்பை உருவாக்கவும்.

 ஒவ்வொரு செலவும் பின்வரும் வடிவில் பிரதிநிதித்துவம் பெறப்படும்:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## கருவிகளை வரையறுத்தல் - மின்னஞ்சலை உருவாக்குதல்

செலவுத்தகவல் கோரிக்கைக்கு மின்னஞ்சலை உருவாக்க ஒரு கருவி செயல்பாட்டை உருவாக்கவும்.
- இந்த கருவி Microsoft Agent Framework இல் இருந்து `@tool` அலங்காரக் காட்டியைப் பயன்படுத்துகிறது.
- செலவுகளின் மொத்தத் தொகையை கணக்கிட்டு, விவரங்களை மின்னஞ்சல் உட்கதையாக வடிவமைக்கிறது.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# ரசீதுக் கோப்புப் படங்களிலிருந்து பயணச் செலவுகளைப் பெறும் கருவி

ரசீதுக் கோப்புப் படங்களிலிருந்து பயணச் செலவுகளைப் பெற ஒரு கருவி செயல்பாட்டை உருவாக்கவும்.
- இந்த கருவி Microsoft Agent Framework இல் இருந்து `@tool` அலங்காரியைப் பயன்படுத்துகிறது.
- இது ரசீதுக் கோப்புப் படத்தைப் படிக்கிறது, அதை base64 ஆக குறியாக்கம் செய்கிறது, மற்றும் முகவரிக்கு பகுப்பாய்வு செய்ய தரவு URI ஐத் திருப்பி வழங்குகிறது.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## செலவுகளை செயலாக்குதல்

எஜன்ட்களை வரையறுக்கவும், அவற்றை `WorkflowBuilder` ஐப் பயன்படுத்தி தொடர்ச்சியான வேலைநிரலாக்கத்தில் இணைக்கவும்.
- OCR எஜன்ட், `load_receipt_image` கருவியைப் பயன்படுத்தி ரசீது படத்திலிருந்து கட்டமைக்கப்பட்ட செலவு தரவை எடுக்கிறது.
- மின்னஞ்சல் எஜன்ட் எடுத்துக்கொண்ட தரவை எடுத்துக்கொண்டு, `generate_expense_email` கருவியைப் பயன்படுத்தி ஒரு தொழில்முறை செலவு கோரிக்கை மின்னஞ்சலை உருவாக்குகிறது.
- `WorkflowBuilder` மற்றும் `add_edge` கொண்டு தொடர்ச்சியான குழாயライン உருவாக்கப்படுகிறது: OCR எஜன்ட் → மின்னஞ்சல் எஜன்ட்.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

வரிசைப்படுத்தப்பட்ட பணிய்முறை கட்டமைக்கவும் அதை இயக்கி ரசீது படத்தை செயலாக்கி செலவு உரிமை இமெயிலை உருவாக்கவும்.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**குறிப்பு**:  
இந்தக் கோப்பு AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) மூலம் மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயன்றாலும், தானியங்கி மொழிபெயர்ப்புகளில் தவறுகள் அல்லது கூர்மைகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். இயல்புநிலை மொழியில் உள்ள அசல் ஆவணம் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கிறோம். இந்த மொழிபெயர்ப்பின் பயன்பாட்டால் ஏற்பட்ட எந்தவொரு தவறுதல்களுக்கும் அல்லது புரிதல் மிதிப்புகளுக்கும் நாங்கள் பொறுப்பேற்கமாட்டோம்.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
